# CMSC5702 Assignment 1 Performance Report

## Steady-state 2D heat distribution

The OpenMP and MPI programs implement the supplied C++ Jacobi solver. Cluster measurements are **not yet collected**. The three timing CSVs currently contain blank runtime fields; the tables show missing values and the plotting cells request measurements until these fields are filled. No performance conclusion is claimed from local correctness tests.

Follow [testing guide](../docs/TESTING.md) to build, check correctness, collect five cluster runs per configuration, and rerun this notebook. All numerical results below are calculated from those CSVs, never from `report_sample.ipynb`.


## 1. Implementation and measurement method

**Sequential.** `src/heat_seq.cpp` is unchanged. It initializes the interior to 25°C, keeps the left wall at 1000°C and the other walls at 0°C, and uses two arrays for Jacobi iteration. The four corners are 0°C. Every version retains the element-by-element copy from `w` to `u` and stops at `diff <= 0.001` or the iteration cap.

**OpenMP.** One persistent thread team computes rows with `schedule(static)`. A maximum reduction combines the per-thread differences. The stencil barrier completes all reads before copying, and the iteration barrier completes the copy and convergence reads before resetting the difference. One thread writes each live snapshot. The runtime's dynamic team resizing is disabled so the reported thread count matches the solver team.

**MPI.** Processes own contiguous interior-row stripes. The first `(m-2) % p` ranks receive one extra row. Each local array has two ghost rows; two `MPI_Sendrecv` calls exchange old boundary rows each iteration. `MPI_Allreduce` computes the global maximum difference. `MPI_Gatherv` assembles rows in global order for previews, live snapshots, and final output. Only rank 0 writes files and console output.

**Timing.** Compile all programs with `-O3` and use the non-LIVE binaries. Time only `find_steady_state()`: the sequential timer is unchanged, OpenMP uses `omp_get_wtime()`, and MPI uses `MPI_Wtime()`. MPI reports the maximum rank time. Its initial barrier, elapsed-time reduction, and final output are outside that interval. This aggregation and disabling LIVE are experimental conventions, not additional course rules. Each configuration has five runs, whose arithmetic mean supplies the plotted time; error bars show the sample standard deviation.

The sequential program supplies every `p=1` point. OpenMP uses one node with bound threads. MPI placement is:

| Processes | Nodes | Processes per node |
|---:|---:|---:|
| 2 | 1 | 2 |
| 4 | 1 | 4 |
| 8 | 2 | 4 |
| 16 | 4 | 4 |
| 32 | 4 | 8 |

Record the cluster node/CPU model and compiler/MPI versions here when measuring. Host allocations and raw run logs are retained in `results/`.


## 2. Load measurements

Install `pandas` and `matplotlib` in the notebook environment. The repository stores `seq_times.csv`, `omp_times.csv`, and `mpi_times.csv` in `data/timings/`. The loading cell supports starting from the repository root or `reports/`, and also supports the flat folder created by `make submission`.

Working CSV columns: `n,p,runtime1,runtime2,runtime3,runtime4,runtime5`. Runtimes are in seconds; a blank means **not measured**, not zero. This schema follows the local notebook's five-run layout and still needs confirmation against the course's official CSV format.

For each grid size, $S(n,p)=T(n,1)/T(n,p)$ and $E(n,p)=S(n,p)/p$. A mean is available only when all five runtimes are present.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

# Works from the repository root, reports/, or the flat submission folder.
for timing_dir in (Path("data/timings"), Path("../data/timings"), Path(".")):
    if (timing_dir / "seq_times.csv").is_file():
        break

run_columns = [f"runtime{i}" for i in range(1, 6)]
seq_df = pd.read_csv(timing_dir / "seq_times.csv")
omp_df = pd.read_csv(timing_dir / "omp_times.csv")
mpi_df = pd.read_csv(timing_dir / "mpi_times.csv")

for df in (seq_df, omp_df, mpi_df):
    df["average_time"] = df[run_columns].mean(axis=1, skipna=False)
    df["std_dev"] = df[run_columns].std(axis=1, skipna=False, ddof=1)

seq_time = seq_df.set_index("n")["average_time"]
seq_std = seq_df.set_index("n")["std_dev"]
for df in (omp_df, mpi_df):
    df["speedup"] = df["n"].map(seq_time) / df["average_time"]
    df["efficiency"] = df["speedup"] / df["p"]

for name, df in (("Sequential", seq_df), ("OpenMP", omp_df), ("MPI", mpi_df)):
    print(name)
    display(df)


Matplotlib is building the font cache; this may take a moment.


Sequential


,n,p,runtime1,runtime2,runtime3,runtime4,runtime5,average_time,std_dev
0,200,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,300,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,400,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


OpenMP


,n,p,runtime1,runtime2,runtime3,runtime4,runtime5,average_time,std_dev,speedup,efficiency
0,200,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,200,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,200,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,200,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,200,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,300,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,300,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,300,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,300,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,300,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


MPI


,n,p,runtime1,runtime2,runtime3,runtime4,runtime5,average_time,std_dev,speedup,efficiency
0,200,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,200,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,200,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,200,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,200,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,300,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,300,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,300,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,300,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,300,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Execution time, speedup, and efficiency

The plots use the sequential program at `p=1`, followed by the parallel measurements at `p=2,4,8,16,32`. A missing measurement remains absent from the corresponding curve.


In [2]:
data = seq_df.dropna(subset=["average_time"]).sort_values("n")
if data.empty:
    print("Sequential: collect all five runtimes per size, then rerun.")
else:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(data["n"], data["average_time"], yerr=data["std_dev"], marker="o", capsize=4)
    ax.set(xlabel="Grid size n", ylabel="Time (seconds)", title="Sequential execution time")
    ax.set_xticks([200, 300, 400])
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


Sequential: collect all five runtimes per size, then rerun.


In [3]:
p_ticks = [1, 2, 4, 8, 16, 32]
for df, name in ((omp_df, "OpenMP"), (mpi_df, "MPI")):
    data = df.dropna(subset=["average_time"])
    if data.empty:
        print(f"{name}: collect all five runtimes per configuration, then rerun.")
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for n, group in data.groupby("n"):
        group = group.sort_values("p")
        p = [1] + group["p"].tolist()
        times = [seq_time[n]] + group["average_time"].tolist()
        errors = [seq_std[n]] + group["std_dev"].tolist()
        baseline = 1.0 if pd.notna(seq_time[n]) else float("nan")
        speedup = [baseline] + group["speedup"].tolist()
        efficiency = [baseline] + group["efficiency"].tolist()
        axes[0].errorbar(p, times, yerr=errors, marker="o", capsize=3, label=f"n={n}")
        axes[1].plot(p, speedup, marker="o", label=f"n={n}")
        axes[2].plot(p, efficiency, marker="o", label=f"n={n}")

    for ax, title, ylabel in zip(axes, ["Execution time", "Speedup", "Efficiency"],
                               ["Time (seconds)", "Speedup", "Efficiency"]):
        ax.set_xscale("log", base=2)
        ax.set_xticks(p_ticks)
        ax.set_xticklabels(p_ticks)
        ax.set(xlabel="Number of processors p", ylabel=ylabel, title=f"{name}: {title}")
        ax.grid(True, alpha=0.3)
    axes[0].legend()
    fig.tight_layout()
    plt.show()


OpenMP: collect all five runtimes per configuration, then rerun.
MPI: collect all five runtimes per configuration, then rerun.


In [4]:
has_data = any(df["average_time"].notna().any() for df in (seq_df, omp_df, mpi_df))
if not has_data:
    print("OpenMP/MPI comparison: cluster measurements are pending.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, n in zip(axes, [200, 300, 400]):
        if pd.notna(seq_time[n]):
            ax.axhline(seq_time[n], color="black", linestyle="--", label="Sequential")
        for df, name in ((omp_df, "OpenMP"), (mpi_df, "MPI")):
            data = df[(df["n"] == n) & df["average_time"].notna()].sort_values("p")
            ax.plot(data["p"], data["average_time"], marker="o", label=name)
        ax.set_xscale("log", base=2)
        ax.set_xticks([2, 4, 8, 16, 32])
        ax.set_xticklabels([2, 4, 8, 16, 32])
        ax.set(xlabel="Number of processors p", ylabel="Time (seconds)", title=f"n={n}")
        ax.grid(True, alpha=0.3)
        ax.legend()
    fig.suptitle("OpenMP and MPI execution time")
    fig.tight_layout()
    plt.show()


OpenMP/MPI comparison: cluster measurements are pending.


## 4. Interpretation to complete after cluster measurements

No cluster runtimes have been collected. For each grid size, report the fastest measured worker count for each implementation, its speedup and efficiency, and compare OpenMP and MPI at the same `p`. Use the five-run variation when interpreting small differences.

The mechanisms to examine are specific to these implementations. OpenMP reduces per-thread row work but retains synchronization and array-copy traffic each iteration. MPI adds two row exchanges and a global maximum reduction each iteration; moving from one node to multiple nodes may change these costs. Increasing `n` increases useful stencil work relative to row-exchange volume. These are explanations to test against the measurements, not observed performance results.

**Measured findings:** pending the cluster runs.


## 5. Correctness and remaining work

Local checks passed on macOS using Clang, libomp, and MPICH: 16 configurations in ordinary and LIVE builds, for 64 parallel-to-sequential comparisons. Console results and serialized arrays matched. All three LIVE programs also produced snapshots matching the sequential reference at iterations 1,000 and 2,000, followed by the matching final array at iteration 2,500. These local checks do not measure cluster performance.

The sequential solver logic is preserved; changes from the original source are formatting only. Local verification compares serialized arrays and console output after excluding only worker-count lines, measured time, and output filenames; worker counts are checked separately. The test plan covers uneven row counts, one row per worker, rectangular grids, sizes around the preview threshold, converged and capped runs, and LIVE snapshots. See [testing guide](../docs/TESTING.md) for repeatable cluster commands.

The code follows the C++ baseline for `<= 0.001`, previews at size 20, and final file creation. These differ from parts of the handout wording as recorded in `docs/REQUIREMENTS.md` section 10.

Cluster correctness checks, the 165 required timing runs, hardware/compiler details, official CSV-format confirmation, and the measured interpretation remain to be completed.


## 6. AI acknowledgement and sources

I acknowledge the use of OpenAI Codex (https://openai.com/codex/) to read the assignment materials, generate the OpenMP and MPI implementations from the supplied sequential source, prepare the Makefile and Slurm benchmark script, adapt the report to CSV inputs, draft the testing guide and report text, run local correctness comparisons, organize the repository, and correct an MPI timing compilation error. This is a record of the tool's contribution, not a claim that the student independently authored or verified these changes.

The handout (page 7) prohibits submitting the assignment specification to an AI tool and asking it to generate all required program code. This draft includes generated program code. The acknowledgement does not establish course-policy compliance; review the permitted scope with the course staff before submission.

Key prompts used:

> Read README.md and complete the whole project. Requirements: no extra functions, only sacrify the basic requirements constrained in README document. No extra sanify check/smoke test in code. As least helpers as possible.

> leave everything prepared for afterwards test. make sure the document for such tests are well-prepared and simple

> organize all the files into structured project repo.

Sources: [assignment1.pdf](../docs/assignment1.pdf), pp. 3–7; [assignment requirements](../docs/REQUIREMENTS.md); the supplied [src/heat_seq.cpp](../src/heat_seq.cpp); and the structure of the original `report.ipynb`. The sample notebook's measurements were not used.
